# Webserv - Résumé du travail réalisé

## Point de départ — Analyse initiale

### Fichiers existants avant l'intégration

| Fichier | État |
|---------|------|
| `Server.hpp / Server.cpp` | Boucle `poll()`, accept, read/write, timeouts |
| `Client.hpp / Client.cpp` | `readData()`, `writeData()`, `reset()`, keep-alive |
| `ConfigParser.cpp` | Parse du fichier `.conf` → `std::vector<ServerConfig>` |
| `Request.hpp / Request.cpp` | Parsing HTTP (méthode, URI, headers, body) |
| `Response.hpp / Response.cpp` | Construction réponse (`setStatus`, `setHeader`, `setBody`, `build`) |
| `Router.hpp / Router.cpp` | Dispatch selon `RouteType` (fichier statique, redirect, erreur) |
| `CGIHandler.hpp / CGIHandler.cpp` | `executeCGI()` bloquant (fork + execve + waitpid) |
| `Dico.hpp` | Structures : `ServerConfig`, `LocationConfig`, `Request`, `Response`, `CGIData`, `ClientData`, `RouteResult` |

### Ce qui manquait

1. **Router** : pas intégré dans `Server.cpp` (réponse hardcodée)
2. **POST / DELETE** : non implémentés
3. **Autoindex** : non implémenté
4. **CGI non-bloquant** : `executeCGI()` bloquait toute la boucle `poll()`
5. **Validations** : `client_max_body_size`, multipart, virtual hosts

## Étape 1 — Intégration du Router dans Server.cpp

### Problème
Le serveur renvoyait une réponse hardcodée (`200 OK Hello World`) au lieu d'utiliser le Router.

### Solution
Remplacement dans `handleClientEvents()` du code hardcodé par :

```cpp
RouteResult result = Router::route(req, client->getServerPort(), _configs);

switch (result.type) {
    case ROUTE_FILE:      Router::serveFile(result.filepath, *res); break;
    case ROUTE_REDIRECT:  Router::handleRedirect(result, *res); break;
    case ROUTE_CGI:       /* executeCGI() */ break;
    case ROUTE_DIRECTORY:  /* 403 par défaut */ break;
    case ROUTE_UPLOAD:    /* POST upload */ break;
    case ROUTE_DELETE:    /* DELETE */ break;
    case ROUTE_ERROR:     Router::makeError(result.error_code, *res); break;
}
```

### Fichiers modifiés
- `srcs/core/Server.cpp` : dispatch via `Router::route()`

## Étape 2 — Implémentation POST (upload) et DELETE

### POST — Upload de fichiers

Ajout dans `Router.cpp` :
```cpp
bool handleUpload(const Request& request, const std::string& uploadDir, Response& response)
```
- Extraction du nom de fichier depuis le header `Content-Disposition` ou l'URI
- Écriture du body dans `uploadDir + filename`
- Réponse `201 Created`

### DELETE — Suppression de fichiers

```cpp
bool handleDelete(const std::string& filepath, Response& response)
```
- Vérifie que le fichier existe (`access()`)
- Supprime avec `remove()`
- Réponse `200 OK` ou `404 Not Found`

### Refactoring du Router

Le fichier `Router.cpp` devenait trop grand → découpage en fonctions par méthode :
- `routeGET()` : fichier statique, redirect, directory
- `routePOST()` : upload, CGI
- `routeDELETE()` : suppression

### Fichiers modifiés
- `srcs/http/Router.cpp` : ajout `handleUpload()`, `handleDelete()`, refactoring
- `includes/http/Router.hpp` : nouvelles déclarations

## Étape 3 — Autoindex (listing de répertoire)

### Principe
Quand `autoindex on` dans la config et qu'une URI pointe vers un répertoire sans `index.html` → génération d'une page HTML listant les fichiers.

### Implémentation

**Nouveau fichier** : `srcs/http/Autoindex.cpp`

```cpp
namespace Router {
bool handleAutoindex(const std::string& dirpath, const std::string& uri, Response& response)
{
    DIR* dir = opendir(dirpath.c_str());
    // Collecte des entrées avec readdir()
    // Tri alphabétique
    // Génération HTML : tableau avec nom, taille, date
    // Liens cliquables basés sur l'URI (pas le chemin filesystem)
    ResponseBuilder::setStatus(response, 200);
    ResponseBuilder::setHeader(response, "Content-Type", "text/html");
    ResponseBuilder::setBody(response, html.str());
    return true;
}
}
```

### Intégration dans Server.cpp
```cpp
case ROUTE_DIRECTORY:
    Router::handleAutoindex(result.filepath, req->uri, *res);
    break;
```

### Fichiers créés/modifiés
- `srcs/http/Autoindex.cpp` (**créé**)
- `includes/http/Router.hpp` : signature mise à jour
- `srcs/core/Server.cpp` : case `ROUTE_DIRECTORY`
- `Makefile` : ajout `http/Autoindex.cpp`

## Étape 4 — CGI non-bloquant

### Problème
`executeCGI()` utilisait `waitpid()` bloquant → le serveur entier était gelé pendant l'exécution d'un script CGI.

### Architecture de la solution

```
CLIENT_READING → parse requête
       ↓
ROUTE_CGI détecté → startCGI() → fork + execve
       ↓
CLIENT_WAITING_CGI → pipe_out ajouté à poll()
       ↓
poll() détecte POLLIN sur pipe → lecture buffer
       ↓
EOF/POLLHUP → finishCGI() → parse output → CLIENT_WRITING
```

### Nouveau fichier : `srcs/cgi/CGIAsync.cpp`

**`startCGI()`** : fork + execve, pipe_out en `O_NONBLOCK`, retourne `CGIData`

**`finishCGI()`** : close pipe, waitpid, parse output CGI, retourne `Response`

### Modifications dans Server.cpp

- **`buildPollFds()`** : ajout des pipes CGI après les sockets clients, `_cgiStartIndex` pour partitionner
- **`handleClientEvents()`** : boucle limitée à `_cgiStartIndex`, `ROUTE_CGI` appelle `startCGI()` au lieu de `executeCGI()`
- **`handleCGIEvents()`** : **nouveau** — itère à partir de `_cgiStartIndex`, lit les pipes, détecte EOF/POLLHUP, appelle `finishCGI()`
- **`checkTimeouts()`** : timeout CGI de 30s → `kill(SIGKILL)` + `waitpid()` + réponse 504
- **`run()`** : ajout de `handleCGIEvents()` dans la boucle principale

### Attributs ajoutés à Server
```cpp
std::map<int, int> _cgiPipeToClient;  // pipe_out fd → client fd
size_t _cgiStartIndex;                // partition dans _pollFds
```

### Fichiers créés/modifiés
- `srcs/cgi/CGIAsync.cpp` (**créé**)
- `includes/cgi/CGIHandler.hpp` : déclarations `startCGI()`, `finishCGI()`, fonctions utilitaires
- `srcs/cgi/CGIHandler.cpp` : suppression du `static` des fonctions partagées
- `includes/core/Server.hpp` : nouveaux attributs et méthode
- `srcs/core/Server.cpp` : refonte du dispatch CGI
- `includes/core/Client.hpp` / `srcs/core/Client.cpp` : getter `getCGIData()`
- `Makefile` : ajout `cgi/CGIAsync.cpp`

## Résultats des tests

| # | Test | Résultat |
|---|------|----------|
| 1 | GET fichier statique (`/index.html`) | 200 OK |
| 2 | GET fichier inexistant (`/nope`) | 404 Not Found |
| 3 | GET redirect (`/old-page`) | 301 Moved Permanently |
| 4 | POST upload (`/uploads/test.txt`) | 201 Created |
| 5 | GET fichier uploadé | 200 OK |
| 6 | DELETE fichier (`/uploads/test.txt`) | 200 OK |
| 7 | GET après DELETE | 404 Not Found |
| 8 | CGI Python (`/cgi-test/hello.py`) | 200 OK |
| 9 | CGI Bash (`/cgi-test/test.sh`) | 200 OK |
| 10 | Méthode non autorisée (PUT) | 405 Method Not Allowed |
| 11 | Autoindex ON (`/uploads/`) | 200 OK (listing HTML) |
| 12 | Autoindex OFF (`/cgi-test/`) | 403 Forbidden |
| 13 | CGI concurrent + requête statique | Statique: 7ms, CGI: 25-87ms (non-bloquant vérifié) |

## Arborescence des fichiers

```
Webserv/
├── includes/
│   ├── core/
│   │   ├── Server.hpp
│   │   └── Client.hpp
│   ├── http/
│   │   ├── Router.hpp
│   │   ├── Request.hpp
│   │   └── Response.hpp
│   ├── config/
│   │   └── ConfigParser.hpp
│   ├── cgi/
│   │   └── CGIHandler.hpp
│   └── utils/
│       └── Dico.hpp
├── srcs/
│   ├── main.cpp
│   ├── core/
│   │   ├── Server.cpp
│   │   └── Client.cpp
│   ├── http/
│   │   ├── Router.cpp
│   │   ├── Request.cpp
│   │   ├── Response.cpp
│   │   └── Autoindex.cpp      ← NOUVEAU
│   ├── config/
│   │   └── ConfigParser.cpp
│   └── cgi/
│       ├── CGIHandler.cpp
│       └── CGIAsync.cpp        ← NOUVEAU
├── config/
│   └── default.conf
├── Makefile
└── Docs/
    └── resume.ipynb
```

## Ce qu'il reste à faire

- [ ] Validation `client_max_body_size` → réponse **413 Request Entity Too Large**
- [ ] Parsing multipart `form-data` pour les uploads
- [ ] Virtual hosts : routing par `server_name` (Host header)
- [ ] Pages d'erreur personnalisées depuis la config
- [ ] Tests de stress / charge
- [ ] Conformité complète HTTP/1.1

## Étape 5 — Corrections issues de la fiche de correction

Après relecture avec la fiche de correction 42, **5 corrections** ont été identifiées et appliquées.

### Correction 1 — Supprimer errno dans `readData()` (Client.cpp)

**Problème** : `readData()` testait `errno` après `recv()`, ce qui est interdit en C++98 (non fiable avec les sockets non-bloquantes).

**Avant** :
```cpp
if (bytesRead < 0)
{
    if (errno == EAGAIN || errno == EWOULDBLOCK)
        return 0;
    std::cout << "ERROR: recv() failed" << std::endl;
    return -1;
}
```

**Après** :
```cpp
if (bytesRead < 0)
{
    std::cout << "ERROR: recv() failed pour fd " << _data.socket_fd << std::endl;
    return -1;
}
```

**Fichier** : `srcs/core/Client.cpp` — `readData()`

---

### Correction 2 — Gestion complète de `writeData()` (Client.cpp)

**Problème** : `writeData()` ne gérait pas les cas `n < 0` (erreur send) et `n == 0` (client déconnecté). Aussi, le message d'erreur disait `"recv()"` au lieu de `"send()"`.

**Avant** :
```cpp
int n = send(_data.socket_fd, data, reste, 0);
if (n > 0) { ... }
return n;
```

**Après** :
```cpp
int n = send(_data.socket_fd, data, reste, 0);
if (n > 0)
{
    _data.response.bytes_sent += n;
    if (_data.response.bytes_sent >= _data.response.send_buffer.size())
        _data.response.is_complete = true;
}
if (n < 0)
{
    std::cout << "ERROR: send() failed pour fd " << _data.socket_fd << std::endl;
    return -1;
}
if (n == 0)
{
    std::cout << "Client fd " << _data.socket_fd << " a ferme la connexion" << std::endl;
    return 0;
}
return n;
```

**Fichier** : `srcs/core/Client.cpp` — `writeData()`

---

### Correction 3 — Lire uniquement en état CLIENT_READING (Server.cpp)

**Vérification** : le code vérifie déjà l'état avant de lire/écrire.

```cpp
// Ligne 238 — lecture protégée
if (client->getClientState() == CLIENT_READING)
{
    int result = client->readData();
    ...
}

// Ligne 390 — écriture protégée
if (client->getClientState() == CLIENT_WRITING)
{
    int result = client->writeData();
    ...
}
```

→ Déjà en place, aucune modification nécessaire.

**Fichier** : `srcs/core/Server.cpp` — `handleClientEvents()`

---

### Correction 4 — Détection des ports dupliqués (ConfigParser.cpp)

**Vérification** : la détection est déjà implémentée dans `validateConfig()`.

```cpp
void ConfigParser::validateConfig() {
    std::set<int> ports;
    if (_servers.empty())
        throw std::runtime_error("No server defined in config");
    for (size_t i = 0; i < _servers.size(); i++) {
        if (!ports.insert(_servers[i].listen_port).second) {
            std::ostringstream oss;
            oss << "Duplicate port: " << _servers[i].listen_port;
            throw std::runtime_error(oss.str());
        }
        validateServer(_servers[i]);
    }
}
```

→ Déjà en place, aucune modification nécessaire.

**Fichier** : `srcs/config/ConfigParser.cpp` — `validateConfig()`

---

### Correction 5 — Bug `client_max_body_size` / 413 (Server.cpp) — BLOQUANT

**Problème** : la condition pour vérifier la taille du body était cassée. Le `||` rendait la condition toujours vraie pour tout POST avec un body, ce qui renvoyait **413 sur tous les POST**.

**Avant** (ligne 278) :
```cpp
else if (req->body.size() || req->content_length > config->max_body_size)
```

**Après** :
```cpp
else if (req->body.size() > config->max_body_size)
```

**Impact** : sans ce fix, les tests POST upload, CGI POST et tout POST renvoyaient 413 au lieu du code attendu.

**Fichier** : `srcs/core/Server.cpp` — `handleClientEvents()`, ligne 278

## Étape 6 — Script de tests d'intégration (35 tests)

Le script `tests/run_tests.sh` a été entièrement réécrit pour couvrir tous les points de la fiche de correction.

### Fonctionnement

- **Auto-compile** le projet avec `make`
- **Démarre** le serveur automatiquement avec `config/default.conf`
- **Exécute** 35 tests répartis en 13 sections
- **Arrête** le serveur proprement (`trap EXIT`)
- **Nettoyage** des fichiers uploadés pendant les tests

### Fonctions de test

| Fonction | Vérifie |
|----------|---------|
| `run_test` | Code HTTP attendu |
| `run_test_header` | Code HTTP + header spécifique (nom + contenu) |
| `run_test_body` | Code HTTP + contenu dans le body de la réponse |

### Les 13 sections (35 tests)

| # | Section | Tests | Vérifie |
|---|---------|-------|---------|
| 1 | GET fichiers statiques | 1-4 | `/`, `/index.html`, 404, page erreur |
| 2 | Redirections | 5 | 301 + header `Location` |
| 3 | POST upload | 6-9 | Upload, lecture contenu, écrasement, re-lecture |
| 4 | DELETE | 10-12 | Suppression, 404 après, fichier inexistant |
| 5 | Autoindex | 13-14 | ON → `<table>`, OFF → 403 |
| 6 | CGI | 15-18 | Python GET, query string, Bash GET, Python POST |
| 7 | Méthodes non autorisées | 19-20 | 405 DELETE sur `/`, 403 POST sur `/cgi-test/` |
| 8 | Headers HTTP | 21-24 | Content-Type, keep-alive, close |
| 9 | client_max_body_size | 25-26 | 413 (6MB > 5M limit), petit body OK |
| 10 | CGI non-bloquant | 27 | CGI + statique en parallèle (<1s) |
| 11 | Second serveur (8081) | 28-29 | GET OK, POST → 405 (GET only) |
| 12 | Edge cases | 30-33 | URI longue, double slash, sans Host, 10 requêtes rapides |
| 13 | Pages d'erreur | 34-35 | 404 custom "Not Found", 405 custom "Method Not Allowed" |

### Test 413 — Détail technique

Le body de 6MB est généré dans un fichier temporaire (pour éviter "Argument list too long" en bash) :
```bash
python3 -c "import sys; sys.stdout.buffer.write(b'X' * (1024 * 1024 * 6))" > /tmp/webserv_bigbody.bin
curl -X POST "$HOST2/" -H "Content-Type: application/octet-stream" --data-binary @/tmp/webserv_bigbody.bin
```
Envoyé sur le port 8081 (limite 5M dans la config).

### Résultat : 35/35 tests passent

**Fichier** : `tests/run_tests.sh`

## Étape 7 — Tests de stress & fuites mémoire

### Siege — Stress test

| Test | Concurrence | Requêtes | Availability | Req/s | Échecs | Temps réponse |
|------|-------------|----------|-------------|-------|--------|---------------|
| 1 | 10 users | 200 | **100.00%** | 2500 | 0 | 0.00s |
| 2 | 50 users | 2,500 | **100.00%** | 2272 | 0 | 0.02s |
| 3 | 100 users | 5,000 | **100.00%** | 1845 | 0 | 0.05s |
| 4 | 200 users | 20,000 | **100.00%** | 2317 | 0 | 0.06s |
| 5 | 255 users | 51,000 | **100.00%** | 2481 | 0 | 0.07s |

**Total : 78,700 requêtes, 0 échec, 100% availability.**

Critère fiche de correction : availability > 99.5% → **100% partout**.

### Valgrind — Fuites mémoire

```
HEAP SUMMARY:
    in use at exit: 0 bytes in 0 blocks
    total heap usage: 4,928 allocs, 4,928 frees, 4,050,568 bytes allocated

All heap blocks were freed -- no leaks are possible

ERROR SUMMARY: 0 errors from 0 contexts
```

| Vérification | Résultat |
|---|---|
| Fuites mémoire | **0 bytes** — aucune fuite |
| Blocs non libérés | **0 blocks** |
| Allocs / Frees | **4,928 / 4,928** (équilibre parfait) |
| Erreurs mémoire | **0 erreurs** |

### Mémoire du processus (sans Valgrind)

| Moment | RSS |
|--------|-----|
| Avant siege | 3768 KB |
| Après 78,700 requêtes | 3776 KB |

Pas de fuite mémoire, la mémoire reste stable.

**Documentation** : `Docs/test_stress_leak.md`

## Étape 8 — Redesign des pages HTML

Toutes les pages HTML ont été redesignées avec un thème sombre unifié (fond `#0f0f0f`, accents `#e94560` et `#5dade2`).

### Pages modifiées

| Fichier | Description |
|---------|-------------|
| `www/index.html` | Page d'accueil — grille de cartes, dot animé "Serveur actif" |
| `www/errors/404.html` | Page erreur 404 — code géant, message, lien retour |
| `www/errors/405.html` | Page erreur 405 — Method Not Allowed |
| `www/errors/413.html` | Page erreur 413 — Payload Too Large (**créé**) |
| `www/errors/500.html` | Page erreur 500 — Internal Server Error (**créé**) |
| `www/cgi-test/test.py` | CGI Python — tableau variables CGI, paramètres GET, body POST |
| `www/cgi-test/test.sh` | CGI Bash — infos système, variables CGI |
| `srcs/http/Autoindex.cpp` | Autoindex — tableau fichiers avec taille et date |

### Design commun

- Font : Segoe UI / Arial / sans-serif
- Background : `#0f0f0f` (body), `#1a1a1a` (cards/sections)
- Header : gradient `#1a1a2e` → `#16213e`, bordure `#0f3460`
- Titres : `#e94560` (rouge/rose)
- Liens : `#5dade2` (bleu) avec hover → fond bleu, texte noir
- Footer : `Webserv` en gris discret

## Résumé des modifications — Vue d'ensemble

### Fichiers modifiés (corrections)

| Fichier | Modification | Correction |
|---------|-------------|------------|
| `srcs/core/Client.cpp` | Suppression errno dans `readData()` | #1 |
| `srcs/core/Client.cpp` | `writeData()` gère `n < 0` et `n == 0`, typo recv→send | #2 |
| `srcs/core/Server.cpp` | Fix `\|\|` → `>` pour `client_max_body_size` (ligne 278) | #5 |

### Fichiers vérifiés (déjà corrects)

| Fichier | Vérification | Correction |
|---------|-------------|------------|
| `srcs/core/Server.cpp` | Lecture uniquement en `CLIENT_READING` | #3 |
| `srcs/config/ConfigParser.cpp` | Détection ports dupliqués dans `validateConfig()` | #4 |

### Fichiers réécrits

| Fichier | Description |
|---------|-------------|
| `tests/run_tests.sh` | Script de tests réécrit — 35 tests, 13 sections |

### Fichiers redesignés (HTML/CGI)

| Fichier | Description |
|---------|-------------|
| `www/index.html` | Page d'accueil redesignée |
| `www/errors/404.html` | Page erreur 404 redesignée |
| `www/errors/405.html` | Page erreur 405 redesignée |
| `www/errors/413.html` | Page erreur 413 créée |
| `www/errors/500.html` | Page erreur 500 créée |
| `www/cgi-test/test.py` | CGI Python redesigné |
| `www/cgi-test/test.sh` | CGI Bash redesigné |
| `srcs/http/Autoindex.cpp` | Autoindex redesigné |

### État final

| Critère | Résultat |
|---------|----------|
| Tests d'intégration | **35/35** |
| Siege (stress test) | **100% availability** (78,700 requêtes) |
| Valgrind (fuites mémoire) | **0 leaks, 0 errors** |
| Corrections fiche 42 | **5/5** |